In [7]:
# Neural Identifier Training with Particle Filters - Double Pendulum System
# UKF-RHONN vs PF-RHONN following Guerra (2022)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

# ============================================================
# 1) True nonlinear system (Double Pendulum)
# ============================================================
def plant_dynamics(x_state, u=0):
    """
    Continuous dynamics for Double Pendulum System.
    x_state = [theta1, theta1_dot, theta2, theta2_dot]
    
    Equations of motion for double pendulum:
    (m1 + m2)*L1*theta1_ddot + m2*L2*theta2_ddot*cos(theta1-theta2) + 
        m2*L2*theta2_dot^2*sin(theta1-theta2) + (m1+m2)*g*sin(theta1) = 0
    
    m2*L2*theta2_ddot + m2*L1*theta1_ddot*cos(theta1-theta2) - 
        m2*L1*theta1_dot^2*sin(theta1-theta2) + m2*g*sin(theta2) = 0
    
    Parameters:
    - m1, m2: masses (kg)
    - L1, L2: lengths (m)
    - g: gravity (9.81 m/s^2)
    - b1, b2: damping coefficients
    """
    # System parameters
    m1, m2 = 1.0, 1.0  # masses (kg)
    L1, L2 = 1.0, 1.0  # lengths (m)
    g = 9.81           # gravity (m/s^2)
    b1, b2 = 0.1, 0.1  # damping coefficients (increased)
    
    theta1, theta1_dot, theta2, theta2_dot = x_state
    
    delta = theta1 - theta2
    
    # Denominators for the coupled equations
    denom1 = (m1 + m2)*L1 - m2*L1*np.cos(delta)**2
    denom2 = (L2/L1)*denom1
    
    # Add small epsilon to prevent division by zero
    eps = 1e-6
    if abs(denom1) < eps:
        denom1 = eps * np.sign(denom1) if denom1 != 0 else eps
    if abs(denom2) < eps:
        denom2 = eps * np.sign(denom2) if denom2 != 0 else eps
    
    # Acceleration of theta1
    theta1_ddot = (m2*L1*theta1_dot**2*np.sin(delta)*np.cos(delta) +
                   m2*g*np.sin(theta2)*np.cos(delta) +
                   m2*L2*theta2_dot**2*np.sin(delta) -
                   (m1+m2)*g*np.sin(theta1) -
                   b1*theta1_dot) / denom1
    
    # Acceleration of theta2
    theta2_ddot = (-m2*L2*theta2_dot**2*np.sin(delta)*np.cos(delta) +
                   (m1+m2)*(g*np.sin(theta1)*np.cos(delta) -
                           L1*theta1_dot**2*np.sin(delta) -
                           g*np.sin(theta2)) -
                   b2*theta2_dot) / denom2
    
    return np.array([theta1_dot, theta1_ddot, theta2_dot, theta2_ddot])

def plant(x_k, u_k, dt=0.01, process_noise_type='gaussian', process_noise_std=1e-2):
    """
    One Euler step of the discrete plant with ADVANCED process noise.
    """
    # Simple Euler integration
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Initialize noise container
    noise = np.zeros_like(x_kp1)

    # --- NOISE GENERATION ---
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)

    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)

    elif process_noise_type == 'student_t':
        df = 3 
        noise = np.random.standard_t(df, size=x_kp1.shape) * process_noise_std

    elif process_noise_type == 'cauchy':
        noise = np.random.standard_cauchy(size=x_kp1.shape) * process_noise_std

    elif process_noise_type == 'bimodal':
        means = np.random.choice([1, -1], size=x_kp1.shape)
        gauss_part = np.random.normal(0, process_noise_std * 0.5, size=x_kp1.shape)
        noise = (means * 2 * process_noise_std) + gauss_part

    elif process_noise_type == 'impulsive':
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)
        prob_spike = 0.05
        mask = np.random.choice([0, 1], size=x_kp1.shape, p=[1-prob_spike, prob_spike])
        spikes = np.random.normal(0, 10 * process_noise_std, size=x_kp1.shape)
        noise += mask * spikes

    else:  # gaussian (Default)
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

# ============================================================
# 2) RHONN structure for Double Pendulum (4 States)
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -10, 10)
    return 1.0 / (1.0 + np.exp(-beta * z))

# ============================================================
# RHONN CONFIGURATIONS FOR DOUBLE PENDULUM (4 states)
# ============================================================

def rhonn_config_1(x_est, scale=0.5):
    """
    Config 1: Minimal - Linear + Sigmoid
    For 4 states: [theta1, theta1_dot, theta2, theta2_dot]
    """
    theta1, theta1_dot, theta2, theta2_dot = x_est
    s1 = sigmoidal(theta1 * scale)
    s2 = sigmoidal(theta1_dot * scale)
    s3 = sigmoidal(theta2 * scale)
    s4 = sigmoidal(theta2_dot * scale)
    
    return np.array([
        s1, s2, s3, s4,           # sigmoids of each state
        theta1, theta2,           # linear angles
        1.0                       # bias
    ])

def rhonn_config_2(x_est, scale=0.5):
    """
    Config 2: Standard - Interactions + Quadratic
    Includes coupling terms between pendulums
    """
    theta1, theta1_dot, theta2, theta2_dot = x_est
    s1 = sigmoidal(theta1 * scale)
    s2 = sigmoidal(theta1_dot * scale)
    s3 = sigmoidal(theta2 * scale)
    s4 = sigmoidal(theta2_dot * scale)
    
    return np.array([
        s1, s2, s3, s4,           # sigmoids
        s1 * s3,                  # coupling between angles
        s2 * s4,                  # coupling between velocities
        s1**2, s3**2,             # quadratic terms
        theta1, theta2,           # linear angles
        1.0                       # bias
    ])

def rhonn_config_3(x_est, scale=0.5):
    """
    Config 3: Extended - More nonlinear terms
    Full coupling and higher order terms
    """
    theta1, theta1_dot, theta2, theta2_dot = x_est
    s1 = sigmoidal(theta1 * scale)
    s2 = sigmoidal(theta1_dot * scale)
    s3 = sigmoidal(theta2 * scale)
    s4 = sigmoidal(theta2_dot * scale)
    
    delta = theta1 - theta2
    s_delta = sigmoidal(delta * scale)
    
    return np.array([
        s1, s2, s3, s4,           # sigmoids
        s1 * s3,                  # angle coupling
        s2 * s4,                  # velocity coupling
        s_delta,                  # sigmoid of angle difference
        s1**2, s3**2,             # quadratic angles
        s2**2, s4**2,             # quadratic velocities
        theta1, theta2,           # linear angles
        1.0                       # bias
    ])

# Dictionary of available configurations
RHONN_CONFIGS = {
    1: {'func': rhonn_config_1, 'name': 'Minimal (Linear + Sigmoid)', 'n_features': 7},
    2: {'func': rhonn_config_2, 'name': 'Standard (Interactions + Quadratic)', 'n_features': 11},
    3: {'func': rhonn_config_3, 'name': 'Extended (Full Coupling)', 'n_features': 14},
}

# Global variable to store selected configuration
SELECTED_RHONN_CONFIG = None

def set_rhonn_config(config_id=2):
    """Set the active RHONN configuration"""
    global SELECTED_RHONN_CONFIG
    if config_id not in RHONN_CONFIGS:
        raise ValueError(f"Invalid config_id: {config_id}. Choose from {list(RHONN_CONFIGS.keys())}")
    
    SELECTED_RHONN_CONFIG = config_id
    config = RHONN_CONFIGS[config_id]
    print(f"\n{'='*60}")
    print(f"🔧 RHONN Configuration Set:")
    print(f"   ID: {config_id}")
    print(f"   Name: {config['name']}")
    print(f"   Number of features: {config['n_features']}")
    print(f"{'='*60}\n")
    
    return config['n_features']

def construct_z_vector(x_est, scale=0.5):
    """Construct feature vector using the selected RHONN configuration"""
    if SELECTED_RHONN_CONFIG is None:
        raise ValueError("RHONN configuration not set! Call set_rhonn_config() first.")
    
    config_func = RHONN_CONFIGS[SELECTED_RHONN_CONFIG]['func']
    return config_func(x_est, scale)

def print_rhonn_configs():
    """Print all available RHONN configurations"""
    print("\n" + "="*70)
    print("📋 AVAILABLE RHONN CONFIGURATIONS FOR DOUBLE PENDULUM:")
    print("="*70)
    for config_id, config in RHONN_CONFIGS.items():
        print(f"\nConfig {config_id}: {config['name']}")
        print(f"   Features: {config['n_features']}")
        sample_state = np.array([0.5, 1.0, 0.3, 0.8])
        sample_z = config['func'](sample_state, scale=0.5)
        print(f"   Sample z shape: {sample_z.shape}")
    print("="*70 + "\n")

def RHONN_predict(x_state_for_z, w_neuron):
    """Predicts next-state component with a single RHONN neuron."""
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# ============================================================
# 3) UKF trainer for RHONN weights (Guerra, 2022)
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter trainer for RHONN weights.
    Following Guerra (2022) algorithm strictly.
    
    Parameters:
    -----------
    num_neurons : int
        Number of neurons (states to estimate)
    num_weights_per_neuron : int
        Number of weights per neuron (L)
    initial_weights : list of arrays, optional
        Initial weight values
    Q_init : float
        Process noise covariance
    R_init : float
        Measurement noise covariance
    P_init : float
        Initial state covariance
    alpha : float
        UKF parameter (typically 0.001)
    beta : float
        UKF parameter (typically 2 for Gaussian)
    kappa : float
        UKF parameter (typically 0 or 3-L)
    """
    
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-3, R_init=1e-2, P_init=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.L = num_weights_per_neuron  # Dimension of state (weights)
        
        # UKF parameters
        self.alpha = alpha
        self.beta = beta
        self.kappa = kappa if kappa is not None else (3 - self.L)
        
        # Lambda parameter
        self.lambda_param = self.alpha**2 * (self.L + self.kappa) - self.L
        
        # Compute weights for sigma points
        self.compute_sigma_weights()
        
        # Initialize per-neuron parameters
        self.weights, self.P, self.Q, self.R = [], [], [], []
        
        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.05
            
            self.weights.append(w_i)
            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))
    
    def compute_sigma_weights(self):
        """Compute weights for sigma points (Guerra, 2022, Eqs. 15-17)"""
        L = self.L
        lambda_p = self.lambda_param
        
        # Weight for mean (Eq. 15)
        self.Wm = np.zeros(2*L + 1)
        self.Wm[0] = lambda_p / (L + lambda_p)
        
        # Weight for covariance (Eq. 16)
        self.Wc = np.zeros(2*L + 1)
        self.Wc[0] = lambda_p / (L + lambda_p) + (1 - self.alpha**2 + self.beta)
        
        # Remaining weights (Eq. 17)
        for i in range(1, 2*L + 1):
            self.Wm[i] = 1.0 / (2.0 * (L + lambda_p))
            self.Wc[i] = 1.0 / (2.0 * (L + lambda_p))
    
    def generate_sigma_points(self, w_mean, P):
        """
        Generate sigma points (Guerra, 2022, Eq. 18)
        
        χ_0 = w
        χ_i = w + (√((L+λ)P))_i,  i=1,...,L
        χ_i = w - (√((L+λ)P))_{i-L},  i=L+1,...,2L
        """
        L = self.L
        lambda_p = self.lambda_param
        
        # Ensure P is positive definite
        P_safe = 0.5 * (P + P.T)  # Symmetrize
        min_eig = np.min(np.linalg.eigvals(P_safe))
        if min_eig <= 0:
            P_safe += np.eye(L) * (abs(min_eig) + 1e-6)
        
        # Compute matrix square root: sqrt((L+λ)P)
        try:
            sqrt_P = np.linalg.cholesky((L + lambda_p) * P_safe)
        except np.linalg.LinAlgError:
            # Fallback to eigenvalue decomposition
            eigvals, eigvecs = np.linalg.eigh(P_safe)
            eigvals = np.maximum(eigvals, 1e-8)
            sqrt_P = eigvecs @ np.diag(np.sqrt((L + lambda_p) * eigvals)) @ eigvecs.T
        
        # Initialize sigma points
        sigma_points = np.zeros((2*L + 1, L))
        
        # χ_0 = w (Eq. 18a)
        sigma_points[0] = w_mean
        
        # χ_i = w + column_i of sqrt_P, i=1,...,L (Eq. 18b)
        for i in range(L):
            sigma_points[i + 1] = w_mean + sqrt_P[:, i]
        
        # χ_i = w - column_{i-L} of sqrt_P, i=L+1,...,2L (Eq. 18c)
        for i in range(L):
            sigma_points[L + i + 1] = w_mean - sqrt_P[:, i]
        
        return sigma_points
    
    def predict_sigma_points(self, sigma_points):
        """
        Predict sigma points through process model (Guerra, 2022, Eq. 19)
        For weight dynamics: w_{k+1|k} = w_k (static model)
        """
        # Static model: weights don't change deterministically
        return sigma_points.copy()
    
    def predict_measurements(self, sigma_points, z_vector):
        """
        Predict measurements from sigma points (Guerra, 2022, Eq. 22)
        γ_i = h(χ*_i) = z^T χ*_i
        """
        num_sigma = sigma_points.shape[0]
        measurements = np.zeros(num_sigma)
        
        for i in range(num_sigma):
            measurements[i] = np.dot(z_vector, sigma_points[i])
        
        return measurements
    
    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        UKF update following Guerra (2022) algorithm.
        
        Series-Parallel: Use measured true state (chi_k) to build feature vector.
        
        Algorithm steps:
        1. Generate sigma points (Eq. 18)
        2. Predict sigma points (Eq. 19)
        3. Predict mean and covariance (Eqs. 20-21)
        4. Predict measurements (Eq. 22)
        5. Compute innovation statistics (Eqs. 23-25)
        6. Kalman gain (Eq. 26)
        7. Update state and covariance (Eqs. 27-28)
        """
        # Build feature vector from measured state
        z_i = construct_z_vector(chi_k)
        
        for i in range(self.num_neurons):
            # ===== STEP 1: Generate Sigma Points (Eq. 18) =====
            sigma_points = self.generate_sigma_points(self.weights[i], self.P[i])
            
            # ===== STEP 2: Predict Sigma Points (Eq. 19) =====
            sigma_points_pred = self.predict_sigma_points(sigma_points)
            
            # ===== STEP 3: Predict Mean and Covariance (Eqs. 20-21) =====
            # Predicted mean (Eq. 20): w̄_{k+1|k} = Σ W^m_i χ*_i
            w_pred = np.sum(self.Wm[:, np.newaxis] * sigma_points_pred, axis=0)
            
            # Predicted covariance (Eq. 21): P̄_{k+1|k} = Σ W^c_i (χ*_i - w̄)(χ*_i - w̄)^T + Q
            P_pred = self.Q[i].copy()
            for j in range(len(self.Wc)):
                diff = sigma_points_pred[j] - w_pred
                P_pred += self.Wc[j] * np.outer(diff, diff)
            
            # Ensure positive definiteness
            P_pred = 0.5 * (P_pred + P_pred.T)
            if np.min(np.linalg.eigvals(P_pred)) <= 0:
                P_pred += np.eye(self.L) * 1e-6
            
            # ===== STEP 4: Predict Measurements (Eq. 22) =====
            # γ_i = h(χ*_i) = z^T χ*_i
            gamma = self.predict_measurements(sigma_points_pred, z_i)
            
            # ===== STEP 5: Innovation Statistics (Eqs. 23-25) =====
            # Predicted measurement (Eq. 23): ŷ_{k+1|k} = Σ W^m_i γ_i
            y_pred = np.sum(self.Wm * gamma)
            
            # Innovation covariance (Eq. 24): P_yy = Σ W^c_i (γ_i - ŷ)(γ_i - ŷ)^T + R
            P_yy = self.R[i][0]
            for j in range(len(self.Wc)):
                diff_y = gamma[j] - y_pred
                P_yy += self.Wc[j] * diff_y * diff_y
            
            # Ensure P_yy is not too small
            if P_yy < 1e-10:
                P_yy = 1e-10
            
            # Cross-covariance (Eq. 25): P_xy = Σ W^c_i (χ*_i - w̄)(γ_i - ŷ)^T
            P_xy = np.zeros(self.L)
            for j in range(len(self.Wc)):
                diff_x = sigma_points_pred[j] - w_pred
                diff_y = gamma[j] - y_pred
                P_xy += self.Wc[j] * diff_x * diff_y
            
            # ===== STEP 6: Kalman Gain (Eq. 26) =====
            # K = P_xy / P_yy
            K = P_xy / P_yy
            
            # ===== STEP 7: Update (Eqs. 27-28) =====
            # Innovation
            innovation = chi_kp1[i] - y_pred
            
            # Update weights (Eq. 27): w_{k+1} = w̄_{k+1|k} + K(y_{k+1} - ŷ_{k+1|k})
            self.weights[i] = w_pred + K * innovation
            
            # Update covariance (Eq. 28): P_{k+1} = P̄_{k+1|k} - K P_yy K^T
            self.P[i] = P_pred - np.outer(K, K) * P_yy
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            if np.min(np.linalg.eigvals(self.P[i])) <= 0:
                self.P[i] += np.eye(self.L) * 1e-6

# ============================================================
# 4) Particle Filter trainer (Adapted for 4 states)
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle Filter trainer for RHONN weights (FULLY VECTORIZED).
    Adapted for Double Pendulum (4 states).
    """
    
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=500,
                 initial_weights=None, Q_std=0.5, R_std=0.5, ess_threshold=None,
                 enable_optimization=False, opt_learning_rate=0.01, opt_top_k=10):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.enable_optimization = enable_optimization
        self.opt_learning_rate = opt_learning_rate
        self.opt_top_k = min(opt_top_k, n_particles)
        
        # Convert Q_std to per-neuron array
        if np.isscalar(Q_std):
            self.Q_std = np.ones(num_neurons) * Q_std
        else:
            self.Q_std = np.array(Q_std)
            if len(self.Q_std) != num_neurons:
                raise ValueError(f"Q_std must have {num_neurons} elements")
            
        # Convert R_std to variance per-neuron array
        if np.isscalar(R_std):
            self.R_var = np.ones(num_neurons) * (R_std**2)
            self.R_std = np.ones(num_neurons) * R_std
        else:
            R_std_array = np.array(R_std)
            if len(R_std_array) != num_neurons:
                raise ValueError(f"R_std must have {num_neurons} elements")
            self.R_std = R_std_array
            self.R_var = R_std_array**2
            
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        # Initialize particles: (num_neurons, n_particles, num_weights_per_neuron)
        if initial_weights is not None:
            base_weights = np.array([np.copy(initial_weights[i]) if i < len(initial_weights) 
                                    else np.random.randn(num_weights_per_neuron) * 0.05 
                                    for i in range(num_neurons)])
            
            self.particles = (base_weights[:, np.newaxis, :] + 
                            np.random.randn(num_neurons, n_particles, num_weights_per_neuron) * 0.05)
        else:
            self.particles = np.random.randn(num_neurons, n_particles, num_weights_per_neuron) * 0.05

        # Weights: (num_neurons, n_particles)
        self.weights_pf = np.ones((num_neurons, n_particles)) / n_particles

    def _ess_vectorized(self, w):
        """Calculate ESS for all neurons (vectorized)."""
        sum_w = np.sum(w, axis=1, keepdims=True)
        sum_w = np.where(sum_w == 0, 1.0, sum_w)
        w_norm = w / sum_w
        return 1.0 / np.sum(w_norm**2, axis=1)

    def _resample_systematic(self, neuron_index):
        """Systematic resampling."""
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]
        
        w = w / np.sum(w)
        N = len(w)
        
        u0 = np.random.uniform(0.0, 1.0 / N)
        positions = u0 + np.arange(N) / N
        cdf = np.cumsum(w)
        indexes = np.searchsorted(cdf, positions)
        indexes = np.clip(indexes, 0, N - 1)
        
        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N
    
    def _optimize_particles(self, neuron_idx, chi_kp1_i, z):
        """Gradient-based optimization for top-k particles."""
        top_indices = np.argpartition(self.weights_pf[neuron_idx], -self.opt_top_k)[-self.opt_top_k:]
        particles_opt = self.particles[neuron_idx, top_indices]
        
        predictions = particles_opt @ z
        errors = predictions - chi_kp1_i
        gradients = np.outer(errors, z)
        particles_opt -= self.opt_learning_rate * gradients
        
        self.particles[neuron_idx, top_indices] = particles_opt

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """Update particle filter (FULLY VECTORIZED)."""
        z = construct_z_vector(chi_k)

        # ===== PREDICT =====
        noise = np.random.randn(self.num_neurons, self.n_particles, self.num_weights_per_neuron)
        self.particles += noise * self.Q_std[:, np.newaxis, np.newaxis]

        # ===== UPDATE =====
        x_pred_particles = np.einsum('ijk,k->ij', self.particles, z)
        innov = chi_kp1[:, np.newaxis] - x_pred_particles
        ll = -0.5 * (innov**2) / self.R_var[:, np.newaxis]
        ll -= np.max(ll, axis=1, keepdims=True)
        like = np.exp(ll) + 1e-400
        
        self.weights_pf *= like
        self.weights_pf /= np.sum(self.weights_pf, axis=1, keepdims=True)

        # ===== RESAMPLE =====
        ess_all = self._ess_vectorized(self.weights_pf)
        needs_resampling = ess_all < self.ess_threshold
        
        for i in np.where(needs_resampling)[0]:
            self._resample_systematic(i)

        # ===== OPTIMIZATION =====
        if self.enable_optimization:
            for i in range(self.num_neurons):
                self._optimize_particles(i, chi_kp1[i], z)

    def get_estimate(self):
        """Get weight estimates (vectorized)."""
        estimates = np.sum(
            self.weights_pf[:, :, np.newaxis] * self.particles,
            axis=1
        )
        return [estimates[i] for i in range(self.num_neurons)]

    def get_statistics(self):
        """Get statistics (vectorized)."""
        ess = self._ess_vectorized(self.weights_pf)
        stats = {
            'ess': ess.tolist(),
            'ess_ratio': (ess / self.n_particles).tolist(),
            'max_weight': np.max(self.weights_pf, axis=1).tolist(),
            'min_weight': np.min(self.weights_pf, axis=1).tolist()
        }
        return stats
    
    def get_info(self):
        """Get PF configuration information."""
        info = f"\n🔬 Particle Filter Configuration:\n"
        info += f"   Particles: {self.n_particles}\n"
        info += f"   Optimization: {'✅ ENABLED' if self.enable_optimization else '❌ Disabled'}\n"
        if self.enable_optimization:
            info += f"      - Learning Rate: {self.opt_learning_rate}\n"
            info += f"      - Top-K Particles: {self.opt_top_k}\n"
        return info

# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # ============================================================
    # 🔧 SET RHONN CONFIGURATION HERE
    # ============================================================
    RHONN_CONFIG_ID = 2  # Config 2: Standard recommended for double pendulum
    num_weights_per_neuron = set_rhonn_config(RHONN_CONFIG_ID)
    
    # ============================================================
    
    # --- Reproducibility seed ---
    SEED = 7517
    np.random.seed(SEED)
    print(f"🎲 Semilla aleatoria (seed): {SEED}")
    print("   (Para reproducibilidad de resultados)\n")
    
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.02
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    # ⚠️ CORRECTED NOISE PARAMETERS - More realistic
    process_noise_type = 'Laplacian' # Laplacian, uniform, student_t, cauchy, bimodal, impulsive, gaussian
    process_noise_std = 0.05  # Increased from 0.01
    measurement_noise_std = 0.08  # Reduced from 0.3 but still significant

    # Per-state noise parameters for filters
    # Q_std: process noise for weight dynamics (not the plant!)
    Q_std_per_neuron = [0.8, 0.8, 0.8, 0.8]  # theta1, theta1_dot, theta2, theta2_dot
    
    # R_std: measurement noise assumption in filters
    R_std_per_neuron = [0.1, 0.1, 0.1, 0.1]  # Should match measurement_noise_std order

    # --- True system init ---
    x_true = np.zeros((n_steps, 4))
    x_true[0] = [np.pi/6, 0.0, np.pi/4, 0.0]  # Initial angles, zero velocities
    
    # --- RHONN config ---
    num_neurons = 4  # theta1, theta1_dot, theta2, theta2_dot

    num_particles = 1800  # Increased for better performance

    # --- Initial weights ---
    common_initial_weights = [np.random.uniform(-1.0, 1.0, num_weights_per_neuron) 
                              for _ in range(num_neurons)]

    # --- Instantiate Trainers ---
    print("\n" + "="*70)
    print("🚀 INITIALIZING TRAINERS FOR DOUBLE PENDULUM")
    print("="*70)
    
    # UKF Trainer (Guerra, 2022)
    print("\n📘 UKF-RHONN Trainer (Guerra, 2022):")
    print(f"   Alpha: 1e-3, Beta: 2.0, Kappa: {3-num_weights_per_neuron}")
    print(f"   Q_init: 5e-3, R_init: 1e-2, P_init: 2.0")
    ukf_trainer = UKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron, 
        initial_weights=common_initial_weights,
        Q_init=5e-3,  # Increased
        R_init=1e-2,  # Increased
        P_init=2.0,   # Increased
        alpha=1e-3, beta=2.0, kappa=None
    )
    
    # PF Trainer
    print("\n📕 PF-RHONN Trainer:")
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron, n_particles=num_particles, 
        initial_weights=common_initial_weights, 
        Q_std=Q_std_per_neuron, R_std=R_std_per_neuron, 
        ess_threshold=0.5*num_particles,
        enable_optimization=False, 
        opt_learning_rate=0.001*num_particles, 
        opt_top_k=int(0.025*num_particles)
    )

    # Initialize Particle Filter cloud
    for i in range(num_neurons):
        pf_trainer.particles[i] = np.tile(common_initial_weights[i], (pf_trainer.n_particles, 1))
        pf_trainer.particles[i] += np.random.normal(size=(pf_trainer.n_particles, num_weights_per_neuron)) * 0.1

    print(pf_trainer.get_info())
    print("="*70)

    # Storage
    x_hat_ukf = np.zeros((n_steps, 4))
    x_hat_ukf[0] = x_true[0]
    x_hat_pf = np.zeros((n_steps, 4))
    x_hat_pf[0] = x_true[0]

    # Timing variables
    ukf_time_total = 0.0
    pf_time_total = 0.0

    print("\n🎬 Starting Double Pendulum simulation...")
    print(f"   Process noise: {process_noise_type}, std={process_noise_std}")
    print(f"   Measurement noise: Gaussian, std={measurement_noise_std}")
    
    for k in range(n_steps - 1):
        # 1) Evolve true system with PROCESS noise
        x_true[k+1] = plant(x_true[k], 0, dt, process_noise_type, process_noise_std)
        
        # Add MEASUREMENT noise to create observations
        chi_k = x_true[k] + np.random.normal(0, measurement_noise_std, size=4)
        chi_kp1 = x_true[k+1] + np.random.normal(0, measurement_noise_std, size=4)

        # 2) UKF - with timing
        t_start_ukf = time.perf_counter()
        ukf_trainer.update(chi_kp1, chi_k, x_hat_ukf[k])
        z_ukf = construct_z_vector(chi_k)
        for i in range(4):
            x_hat_ukf[k+1, i] = np.dot(ukf_trainer.weights[i], z_ukf)
        t_end_ukf = time.perf_counter()
        ukf_time_total += (t_end_ukf - t_start_ukf)

        # 3) PF - with timing
        t_start_pf = time.perf_counter()
        pf_trainer.update(chi_kp1, chi_k, x_hat_pf[k])
        est_w_pf = pf_trainer.get_estimate()
        z_pf = construct_z_vector(chi_k)
        for i in range(4):
            x_hat_pf[k+1, i] = np.dot(est_w_pf[i], z_pf)
        t_end_pf = time.perf_counter()
        pf_time_total += (t_end_pf - t_start_pf)
        
        if (k+1) % 200 == 0:
            print(f"  ⏳ Paso {k+1}/{n_steps-1} completado")

    print("✅ Simulación Completa.")
    
    # Display timing results
    print("\n" + "="*70)
    print("⏱️  TIEMPOS DE ENTRENAMIENTO:")
    print("="*70)
    print(f"UKF Total:     {ukf_time_total:.4f} segundos  ({ukf_time_total*1000:.2f} ms)")
    print(f"UKF Por Paso:  {ukf_time_total/(n_steps-1)*1000:.4f} ms/paso")
    print(f"\nPF Total:      {pf_time_total:.4f} segundos  ({pf_time_total*1000:.2f} ms)")
    print(f"PF Por Paso:   {pf_time_total/(n_steps-1)*1000:.4f} ms/paso")
    print(f"\nRelación PF/UKF: {pf_time_total/ukf_time_total:.2f}x")
    print("="*70)

    # ============================================================
    # 6) Visualización - Formato Tesis
    # ============================================================

    # Configuración de formato para tesis
    thesis_config = {
        'font_family': 'Computer Modern, serif',
        'font_size': 14,
        'title_font_size': 16,
        'legend_font_size': 13,
        'line_width_true': 2.5,
        'line_width_est': 2.0,
        'plot_width': 1000,
        'plot_height': 500,
        'grid_color': 'rgba(200, 200, 200, 0.3)',
        'grid_width': 0.5
    }

    # Cálculo de MSE por estado
    mse_ukf_per_state = [np.mean((x_true[:, i] - x_hat_ukf[:, i])**2) for i in range(4)]
    mse_pf_per_state = [np.mean((x_true[:, i] - x_hat_pf[:, i])**2) for i in range(4)]
    
    mse_total_ukf = sum(mse_ukf_per_state)
    mse_total_pf = sum(mse_pf_per_state)

    # Reporte MSE
    print("\n" + "="*70)
    print("🏆 MEJOR FILTRO: ", end="")
    mse_dict = {'UKF': mse_total_ukf, 'PF': mse_total_pf}
    best_filter = min(mse_dict, key=mse_dict.get)
    print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.6f})")
    for other_filter, mse_value in mse_dict.items():
        if other_filter != best_filter:
            improvement = ((mse_value - mse_dict[best_filter]) / mse_value) * 100
            print(f"{other_filter} MSE total: {mse_value:.6f} (+{improvement:.2f}% error)")
    print("="*70)

    print("\n--- Comparación de Desempeño (MSE) - Péndulo Doble ---")
    state_names = ['θ₁', 'θ̇₁', 'θ₂', 'θ̇₂']
    for i, name in enumerate(state_names):
        print(f"UKF MSE {name}:  {mse_ukf_per_state[i]:.6f}")
        print(f"PF  MSE {name}:  {mse_pf_per_state[i]:.6f}")
        print()

    # Gráficas individuales por estado
    states_info = [
        {'idx': 0, 'var': 'θ₁', 'desc': 'Ángulo Péndulo 1', 'y_label': 'θ₁ (rad)'},
        {'idx': 1, 'var': 'θ̇₁', 'desc': 'Velocidad Angular Péndulo 1', 'y_label': 'θ̇₁ (rad/s)'},
        {'idx': 2, 'var': 'θ₂', 'desc': 'Ángulo Péndulo 2', 'y_label': 'θ₂ (rad)'},
        {'idx': 3, 'var': 'θ̇₂', 'desc': 'Velocidad Angular Péndulo 2', 'y_label': 'θ̇₂ (rad/s)'}
    ]

    for state_info in states_info:
        i = state_info['idx']
        
        fig = go.Figure()
        
        # Estado real
        fig.add_trace(go.Scatter(
            x=t_history, y=x_true[:, i],
            mode='lines',
            name='Estado Real',
            line=dict(color='#000000', width=thesis_config['line_width_true']),
            showlegend=True
        ))
        
        # Estimación UKF
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_ukf[:, i],
            mode='lines',
            name='UKF-RHONN',
            line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
            showlegend=True
        ))
        
        # Estimación PF
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_pf[:, i],
            mode='lines',
            name='PF-RHONN',
            line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
            showlegend=True
        ))
        
        fig.update_layout(
            title={
                'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Péndulo Doble',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title='Tiempo (s)',
            yaxis_title=state_info['y_label'],
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                x=0.98,
                y=0.98,
                xanchor='right',
                yanchor='top',
                bgcolor='rgba(255, 255, 255, 0.95)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_height'],
            margin=dict(l=80, r=40, t=80, b=60)
        )
        
        fig.show()

    # Espacio de fase 2D para cada péndulo
    for pendulum_idx, pendulum_name in [(0, "Péndulo 1"), (2, "Péndulo 2")]:
        fig_phase = go.Figure()

        fig_phase.add_trace(go.Scatter(
            x=x_true[:, pendulum_idx], y=x_true[:, pendulum_idx+1],
            mode='lines',
            name='Trayectoria Real',
            line=dict(color='#000000', width=3)
        ))

        fig_phase.add_trace(go.Scatter(
            x=x_hat_ukf[:, pendulum_idx], y=x_hat_ukf[:, pendulum_idx+1],
            mode='lines',
            name='Estimación UKF-RHONN',
            line=dict(color='#1f77b4', width=2)
        ))

        fig_phase.add_trace(go.Scatter(
            x=x_hat_pf[:, pendulum_idx], y=x_hat_pf[:, pendulum_idx+1],
            mode='lines',
            name='Estimación PF-RHONN',
            line=dict(color='#d62728', width=2)
        ))

        theta_label = f'θ{pendulum_idx//2 + 1}'
        fig_phase.update_layout(
            title={
                'text': f'Espacio de Fases - {pendulum_name} ({theta_label} vs {theta_label}̇)',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title=f'{theta_label} (rad)',
            yaxis_title=f'{theta_label}̇ (rad/s)',
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                x=0.98,
                y=0.98,
                xanchor='right',
                yanchor='top',
                bgcolor='rgba(255, 255, 255, 0.95)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_width'] * 0.75,
            margin=dict(l=80, r=40, t=80, b=60)
        )

        fig_phase.show()

    # Gráfica de barras comparando MSE
    fig_mse = go.Figure()

    filters = ['UKF-RHONN', 'PF-RHONN']
    colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

    for i, name in enumerate(state_names):
        fig_mse.add_trace(go.Bar(
            name=f'Estado {name}',
            x=filters,
            y=[mse_ukf_per_state[i], mse_pf_per_state[i]],
            marker_color=colors[i],
            text=[f'{mse_ukf_per_state[i]:.2e}', f'{mse_pf_per_state[i]:.2e}'],
            textposition='outside'
        ))

    fig_mse.update_layout(
        title={
            'text': 'Comparación de Error Cuadrático Medio (MSE) - Péndulo Doble',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tipo de Filtro',
        yaxis_title='Error Cuadrático Medio (MSE)',
        yaxis=dict(
            type='log',
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.98,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.95)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )

    fig_mse.show()

    print("\n" + "="*70)
    print("✅ SIMULACIÓN COMPLETADA")
    print("="*70)
    print(f"\nResultados guardados para {n_steps} pasos de simulación")
    print(f"Configuración RHONN: {RHONN_CONFIGS[RHONN_CONFIG_ID]['name']}")
    print(f"Tipo de ruido de proceso: {process_noise_type}")
    print(f"Ganador: {best_filter} con MSE total de {mse_dict[best_filter]:.6f}")
    print("="*70)


🔧 RHONN Configuration Set:
   ID: 2
   Name: Standard (Interactions + Quadratic)
   Number of features: 11

🎲 Semilla aleatoria (seed): 7517
   (Para reproducibilidad de resultados)


🚀 INITIALIZING TRAINERS FOR DOUBLE PENDULUM

📘 UKF-RHONN Trainer (Guerra, 2022):
   Alpha: 1e-3, Beta: 2.0, Kappa: -8
   Q_init: 5e-3, R_init: 1e-2, P_init: 2.0

📕 PF-RHONN Trainer:

🔬 Particle Filter Configuration:
   Particles: 1800
   Optimization: ❌ Disabled


🎬 Starting Double Pendulum simulation...
   Process noise: Laplacian, std=0.05
   Measurement noise: Gaussian, std=0.08
  ⏳ Paso 200/999 completado
  ⏳ Paso 400/999 completado
  ⏳ Paso 600/999 completado
  ⏳ Paso 800/999 completado
✅ Simulación Completa.

⏱️  TIEMPOS DE ENTRENAMIENTO:
UKF Total:     0.6442 segundos  (644.22 ms)
UKF Por Paso:  0.6449 ms/paso

PF Total:      1.2049 segundos  (1204.93 ms)
PF Por Paso:   1.2061 ms/paso

Relación PF/UKF: 1.87x

🏆 MEJOR FILTRO: UKF (MSE total: 0.023318)
PF MSE total: 0.026183 (+10.94% error)

--- Com


✅ SIMULACIÓN COMPLETADA

Resultados guardados para 1000 pasos de simulación
Configuración RHONN: Standard (Interactions + Quadratic)
Tipo de ruido de proceso: Laplacian
Ganador: UKF con MSE total de 0.023318
